# Introduction to Redis with Python (redis-py)

This notebook will guide you through the basic functionalities of `redis-py`, the official Python client for Redis, a key-value store database. (On PyPI the package is simply called `redis`.)

The notebook is written for **Google Colab**, where we install and start a real Redis server. If you run it somewhere else and no Redis server is running, it automatically uses [`fakeredis`](https://pypi.org/project/fakeredis/), a Python imitation of Redis that supports the same commands.

## Installing the libraries and starting the server

In [ ]:
%pip install -q redis fakeredis

In [ ]:
import subprocess, sys, time

# On Google Colab we can install and start a real Redis server.
# Elsewhere we use a Redis server running on your computer, if there is one (see the next cell).
if "google.colab" in sys.modules:
    subprocess.run("apt-get -qq install -y redis-server > /dev/null", shell=True, check=True)
    subprocess.run("redis-server --daemonize yes", shell=True, check=True)
    time.sleep(1)

In the code above, the `--daemonize yes` option ensures the Redis server runs as a background process (detached from the terminal), allowing the notebook to continue without being blocked.

## Connecting to Redis

Now that the Redis server is running, let's connect to it. `decode_responses=True` tells the client to give us Python strings instead of raw bytes, so we don't have to call `.decode('utf-8')` on every value.

In [ ]:
import redis
import fakeredis

_fake_server = fakeredis.FakeServer()

def connect(db=0):
    """Connect to the Redis server on this machine, or to fakeredis if no server is running."""
    r = redis.Redis(host="localhost", port=6379, db=db, decode_responses=True)
    try:
        r.ping()
        return r
    except redis.ConnectionError:
        return fakeredis.FakeRedis(server=_fake_server, db=db, decode_responses=True)

r = connect()

# Test the connection
print("Connected to:", type(r).__name__, "| ping:", r.ping())

In the code above, the `db=0` parameter specifies which logical database to use. Redis supports multiple databases, and these are numbered from 0 to 15 by default. When you connect to Redis, you are connected to database 0 by default unless you specify otherwise.

- You can change the database by setting the db parameter to another number, for example db=1 to use the second database, and so on up to db=15.

Each logical database in Redis is completely independent from the others, so keys stored in db=0 won’t be available in db=1, for example. It’s like having multiple separate Redis databases within a single instance.

> You may find older tutorials that use `redis.StrictRedis`. Since redis-py 3.0, `StrictRedis` is just another name for `redis.Redis`, so they behave exactly the same.

## Inserting a Single Key-Value Pair

Let's start by inserting a single key-value pair into the Redis database.

In [ ]:
# Inserting a single key-value pair
r.set('name', 'John Doe')

# Retrieving the value
name = r.get('name')
print(f"Stored name: {name}")

## Batch Insert

Redis supports batch operations to insert multiple key-value pairs efficiently. We'll use the `mset` method for this.

In [ ]:
# Batch insert key-value pairs
batch_data = {
    'age': 30,
    'city': 'New York',
    'occupation': 'Engineer'
}

r.mset(batch_data)

# Retrieving the values to verify
for key in batch_data.keys():
    print(f"{key}: {r.get(key)}")

Note that `age` comes back as the string `'30'`: Redis stores strings, and it is up to the application to convert them back to numbers.

# Batch insert: Using hset vs mset

Redis provides two useful commands for storing multiple fields:

- `mset`: Used to set multiple independent key-value pairs.
- `hset`: Used to set multiple fields within a single key (specifically for hashes).

### 1. Using mset: Storing Multiple Independent Key-Value Pairs

With `mset`, you can store unrelated key-value pairs in Redis. Each key is independent and holds its own value. To keep things tidy, let's use another logical database (db=2):

In [ ]:
rb = connect(db=2)

In [ ]:
# Storing multiple independent key-value pairs
rb.mset({
    'user:1': 'John Doe',
    'user:2': 'Jane Smith',
    'user:3': 'Alice Johnson'
})

# Retrieving values
print(rb.get('user:1'))
print(rb.get('user:2'))
print(rb.get('user:3'))

Key Characteristics of mset:

- Each key is independent.
- It is used when the data stored under each key doesn’t need to be grouped.

### 2. Using hset: Storing Multiple Fields Within a Single Key

`hset` is used for Redis hashes, where you want to store multiple fields (like a dictionary) under a single key. (You may also see `hmset` in older code: it is deprecated, and `hset` does the same job.)

In [ ]:
# Storing data for a person using hset
rb.hset('person:1', 'name', 'John Doe')
rb.hset('person:1', 'age', '30')
rb.hset('person:1', 'city', 'New York')
rb.hset('person:1', 'occupation', 'Engineer')

# Retrieving all fields of the hash
person_data = rb.hgetall('person:1')
print(person_data)

Calling `hset` once per field works, but every call is a separate round trip to the server. The better way is to pass the whole dictionary at once with `mapping=`:

In [ ]:
# Dictionary to store
cat_data = {
    'name': 'Chopin',
    'age': '3',
    'city': 'Milan',
    'occupation': 'Cat'
}

# Store the whole dictionary in Redis under the key 'cat:1', with a single command
rb.hset('cat:1', mapping=cat_data)

# Retrieve and print the stored data
print(rb.hgetall('cat:1'))

# Read a single field
print(rb.hget('cat:1', 'name'))

Key Characteristics of hset:

- All fields (like name, age, city) are stored under one key (`person:1` in this case).
- This is useful for grouping related data (e.g., storing user profiles or product details).
- You can read or update one field without touching the others.

Now let's get all the keys of db=2

In [ ]:
# Get all keys in the Redis database
for key in rb.keys('*'):
    print(key)

> `KEYS *` is fine for a small database like this one, but on a production server with millions of keys it blocks Redis until it has scanned all of them. In real applications use `scan_iter`, which returns the keys a few at a time:
> ```python
> for key in rb.scan_iter('user:*'):
>     print(key)
> ```

## Deleting a Key-Value Pair

You can delete a key-value pair from the Redis database using the `delete` method.

In [ ]:
# Deleting a key-value pair
r.delete('occupation')

# Trying to retrieve the deleted key
occupation = r.get('occupation')
print(f"Occupation after deletion: {occupation}")

## Additional Redis Features

Redis supports many additional features, such as setting a TTL (time-to-live) for a key or incrementing numeric values.

### Setting TTL
You can set a TTL for a key, after which the key will automatically expire.

In [ ]:
import time

# Setting a key with a TTL of 3 seconds
r.set('temp_key', 'This is temporary', ex=3)
print(f"temp_key: {r.get('temp_key')} (expires in {r.ttl('temp_key')} s)")

# After 3 seconds, the key will expire
time.sleep(3.5)

temp_key = r.get('temp_key')
print(f"temp_key after 3 seconds: {temp_key}")

### Incrementing Numeric Values

Redis allows you to increment numeric values atomically, which is useful in many scenarios such as counters.

In [ ]:
# Incrementing a numeric value
r.set('counter', 0)
r.incr('counter')
print(f"Counter: {r.get('counter')}")

# Incrementing by 5
r.incrby('counter', 5)
print(f"Counter after incrementing by 5: {r.get('counter')}")

### Transactions: Several Commands as One Unit

Each Redis command is atomic, but a sequence of commands is not: another client could run its commands in between. With a **transaction** (`MULTI` / `EXEC`) Redis queues the commands and runs them all together, without anything else in the middle. In redis-py a transaction is created with `pipeline()`:

In [ ]:
r.set('balance:alice', 100)
r.set('balance:bob', 50)

# Move 30 from Alice to Bob: both commands are executed together
with r.pipeline(transaction=True) as pipe:
    pipe.decrby('balance:alice', 30)
    pipe.incrby('balance:bob', 30)
    pipe.execute()

print("Alice:", r.get('balance:alice'), "| Bob:", r.get('balance:bob'))